# 03 — Heston calibration to listed SPX surface

Two-stage calibration: differential evolution finds the basin, L-BFGS-B refines. Target: in-sample IV RMSE < 1.5 vol points on liquid strikes.

## Context

Heston calibration minimizes the weighted RMSE of model versus market implied volatilities across the surface. We work in IV space rather than price space because IV errors are roughly homogeneous across moneyness, whereas price MSE is dominated by ITM quotes and ignores the wings — exactly where stochastic-vol matters.

The calibrator runs in two stages: SciPy `differential_evolution` to find the right basin, then L-BFGS-B for local polish. The Feller condition $2\kappa\theta \ge \xi^2$ is reported as a diagnostic but not enforced — calibrated SPX surfaces routinely violate it, and the QE Monte Carlo scheme handles sub-Feller variance paths by construction.

In [ ]:
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

from volengine.backtesting import load_option_chain, filter_for_calibration
from volengine.calibration import IVQuote, calibrate_heston
from volengine.surfaces import implied_vol
from volengine.models.heston import heston_vanilla_price

In [ ]:
snapshot = load_option_chain('SPY', dt.date.today(), provider='yfinance')
filtered = filter_for_calibration(snapshot, moneyness_band=(0.85, 1.15))
quotes = []
for _, row in filtered.iterrows():
    iv = implied_vol(row['mid'], snapshot.spot, row['strike'],
                     row['dte_years'], snapshot.r, snapshot.q, 'call')
    if np.isfinite(iv):
        quotes.append(IVQuote(K=row['strike'], T=row['dte_years'], iv_mkt=iv,
                              weight=1.0 / max(row['ask'] - row['bid'], 1e-4)))
print(f'{len(quotes)} quotes feed calibration.')
result = calibrate_heston(quotes, S0=snapshot.spot, r=snapshot.r, q=snapshot.q)
result

## Fit quality

**Figure.** Market IVs (markers) vs. calibrated Heston model IVs (lines) on a single trading day, grouped by maturity. A good calibration sits inside the bid-ask half-spread band on liquid strikes; deviations on the deep wings are expected and concentrate the model's known weaknesses.

In [ ]:
# Diagnostic plot: market IV vs model IV per maturity slice.
by_T = {}
for q in quotes:
    by_T.setdefault(q.T, []).append(q)
n_T = len(by_T)
fig, axes = plt.subplots(1, n_T, figsize=(4 * n_T, 4), sharey=True)
axes = [axes] if n_T == 1 else axes
for ax, (T, group) in zip(axes, sorted(by_T.items())):
    Ks = np.array([g.K for g in group])
    iv_mkt = np.array([g.iv_mkt for g in group])
    prices_model = heston_vanilla_price(Ks, T, snapshot.spot, snapshot.r, snapshot.q, result.params)
    iv_model = [implied_vol(float(P), snapshot.spot, K, T, snapshot.r, snapshot.q, 'call')
                for P, K in zip(np.atleast_1d(prices_model), Ks)]
    ax.scatter(Ks, iv_mkt, s=15, label='market')
    ax.plot(Ks, iv_model, 'r-', label='Heston')
    ax.set_title(f'T = {T:.3f}y')
    ax.legend()
fig.suptitle(f'Calibration RMSE: {result.rmse_vol_points*100:.2f} vol pts')
fig.tight_layout()
fig.savefig('../results/figures/heston_calibration.png', dpi=120)
plt.show()

## Parameter sanity checks

Calibrated SPX parameters typically land in the ranges: $\kappa \in [0.5, 5]$, $\theta \in [0.02, 0.08]$, $\xi \in [0.3, 1.5]$, $\rho \in [-0.9, -0.5]$, $v_0$ close to current realized variance. Values outside these ranges usually mean the optimizer got stuck — re-run with a different DE seed or tighter bounds.